# 09 — HMM Extension for Ripple Inference

**Strategy reference:** §9.10 (HMM Framing), §23–§25 (Later
extensions).

V1 Ripple uses a deterministic `ScoreBasedInference` to map evidence
to one of {STABLE, ABSORPTION, EXHAUSTION, WITHDRAWAL, REFILL}.
V2+ replaces this with a **Gaussian-emission Hidden Markov Model**
trained on V1 backtest output.

Latent states: $z_t \in \{1,\dots,K\}$ with
* transition matrix $\mathbf{A} \in \mathbb{R}^{K\times K}$
* emission $\mathbf{x}_t \mid z_t = k \sim \mathcal N(\mu_k, \Sigma_k)$
* observation vector
  $\mathbf x_t = [I_t,\, \text{OFI}_t,\, \text{CVD\_slope}_t,\, \lambda_t,\, W_p,\, F_t]^\top$

In [ ]:
# ── Data-source configuration ─────────────────────────────────────────
# OHLCV (Parquet) — S3 or local, controlled by DATA_STORE env var:
#   Local (default):  reads <project_root>/data/ohlcv/...
#   S3:               uncomment the two lines below
# import os
# os.environ["DATA_STORE"] = "s3"
# os.environ["S3_BUCKET"]  = "trading-data-centheos"
#
# Tick data (HDF5) — always stored locally; pull from S3 on demand:
#   load_ticks(...)              → use local cache (fast, no network)
#   load_ticks(..., refresh=True) → sync from S3 then read (ETag-gated)
#   Requires: AWS_PROFILE=trading (or AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY)
import os
os.environ["AWS_PROFILE"] = "trading"
os.environ["S3_BUCKET"]   = "trading-data-centheos"
# ─────────────────────────────────────────────────────────────────────

import sys, importlib
from pathlib import Path

_here = Path.cwd().resolve()
for _cand in [_here, *_here.parents]:
    if (_cand / "schemas.py").exists():
        _root = _cand; break
else:
    raise RuntimeError("Could not locate project root (no schemas.py found)")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import notebooks.utils as _utils_mod
importlib.reload(_utils_mod)   # always pick up on-disk changes without restarting the kernel

from notebooks.utils import (
    load_ohlcv, list_ohlcv, load_ticks, latest_book,
    plot_ohlcv, plot_equity_curve, configure_pandas, env_summary,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

configure_pandas()
%matplotlib inline

In [ ]:
from hmm.hmm_model import HMMModel
from hmm.hmm_trainer import HMMTrainer, OBS_DIM
print('observation dimension:', OBS_DIM)

## 1. Synthetic ground-truth sequence
Generate observations from a known 4-state HMM so we can verify
the trainer recovers something close to ground truth.  In real
use, observations come from V1 backtest dumps.

In [ ]:
rng = np.random.default_rng(42)
K_true = 4
T      = 4_000

A_true = np.array([
    [0.92, 0.04, 0.02, 0.02],
    [0.05, 0.85, 0.05, 0.05],
    [0.04, 0.06, 0.84, 0.06],
    [0.03, 0.05, 0.04, 0.88],
])
mu_true = np.array([
    [ 0.0,  0.0,  0.0, 0.5, 0.2, 0.0],
    [+0.6, +0.4, +0.5, 0.6, 0.8, 0.7],
    [-0.4, -0.3, -0.5, 0.4, 0.3, 0.2],
    [+0.2, -0.5, +0.1, 1.0, 0.1, 0.5],
])
var_true = np.full((K_true, OBS_DIM), 0.08)

states = np.zeros(T, dtype=int)
obs    = np.zeros((T, OBS_DIM))
states[0] = rng.integers(K_true)
obs[0]    = rng.normal(mu_true[states[0]], np.sqrt(var_true[states[0]]))
for t in range(1, T):
    states[t] = rng.choice(K_true, p=A_true[states[t-1]])
    obs[t]    = rng.normal(mu_true[states[t]], np.sqrt(var_true[states[t]]))
print('synthetic obs shape:', obs.shape, 'state hist:', np.bincount(states))

## 2. Fit a single K

In [ ]:
trainer = HMMTrainer(max_iter=40, tol=1e-4)
model_K4 = trainer.fit(obs, K=K_true)
print('log-likelihood :', model_K4.log_likelihood)
print('BIC            :', model_K4.bic)
print('transition (learned, rounded):')
pd.DataFrame(np.round(model_K4.transition, 3))

### 2.1 Compare learned vs true transitions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
for ax, M, title in [(axes[0], A_true, 'true A'),
                       (axes[1], model_K4.transition, 'learned A')]:
    im = ax.imshow(M, vmin=0, vmax=1, cmap='Blues')
    ax.set_title(title)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f'{M[i,j]:.2f}', ha='center', va='center', fontsize=8)
plt.tight_layout(); plt.show()

## 3. Model selection via BIC (§9.10)
Train K ∈ {3, 4, 5, 6} and pick the lowest BIC.  Synthetic data
should bottom out at K = 4.

In [ ]:
results = []
for K in [3, 4, 5, 6]:
    m = trainer.fit(obs, K=K)
    results.append({'K': K, 'log_lik': m.log_likelihood, 'bic': m.bic})
    print(f'K={K}  ll={m.log_likelihood:.1f}  bic={m.bic:.1f}')
bic_df = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(bic_df['K'], bic_df['bic'], marker='o')
ax.set_xticks(bic_df['K']); ax.set_xlabel('K'); ax.set_ylabel('BIC (lower is better)')
ax.set_title('BIC vs number of HMM states'); ax.grid(alpha=0.3); plt.show()

## 4. Forward-algorithm filtering (intuition)
Implement the recursion in plain NumPy to visualise the state
posterior $\gamma_t(k)$ over time.

In [ ]:
def forward_posterior(model, obs):
    K = model.K
    T = obs.shape[0]
    log_A = np.log(model.transition + 1e-300)
    log_pi = np.full(K, -np.log(K))
    log_alpha = np.zeros((T, K))
    diff = obs[:, None, :] - model.means[None, :, :]
    log_B = (-0.5 * (OBS_DIM * np.log(2 * np.pi)
                      + np.log(model.variances).sum(axis=1)
                      + np.sum(diff * diff / model.variances, axis=2)))
    log_alpha[0] = log_pi + log_B[0]
    for t in range(1, T):
        log_alpha[t] = log_B[t] + np.array([
            np.logaddexp.reduce(log_alpha[t-1] + log_A[:, k]) for k in range(K)
        ])
    gamma = np.exp(log_alpha - np.logaddexp.reduce(log_alpha, axis=1, keepdims=True))
    return gamma

gamma = forward_posterior(model_K4, obs[:500])
fig, ax = plt.subplots(figsize=(12, 3))
for k in range(model_K4.K):
    ax.plot(gamma[:, k], label=f'P(z={k})', linewidth=0.8)
ax.set_title('State posterior γ_t over first 500 observations')
ax.set_xlabel('t'); ax.set_ylabel('γ_t(k)')
ax.legend(loc='upper right'); ax.grid(alpha=0.3); plt.show()

## 5. Emission means heatmap
Each row is a learned state, each column is one of the six
Ripple observables.  States with very different means are easy to
distinguish; clusters of similar rows suggest K is too large.

In [ ]:
obs_names = ['I_t', 'OFI', 'CVD_slope', 'λ', 'W_p', 'F_t']
fig, ax = plt.subplots(figsize=(8, 3.5))
im = ax.imshow(model_K4.means, aspect='auto', cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(OBS_DIM)); ax.set_xticklabels(obs_names)
ax.set_yticks(range(model_K4.K)); ax.set_yticklabels([f'state {k}' for k in range(model_K4.K)])
for i in range(model_K4.K):
    for j in range(OBS_DIM):
        ax.text(j, i, f'{model_K4.means[i,j]:+.2f}', ha='center', va='center', fontsize=8)
ax.set_title('Learned emission means'); plt.colorbar(im); plt.show()

## 6. Production path

* Train offline in Python (this notebook), save via `HMMModel.save`.
* `HMMBasedInference::load_model_from_string()` in C++ ingests the
  JSON and runs the forward algorithm per event ($O(K^2)$ per
  step, ≤25 multiply-adds at K=5 — well inside the §21 latency
  budget).
* `hmm.abtest` provides the A/B harness used in CI to confirm the
  HMM backend doesn't regress vs `ScoreBasedInference`.
